# AMOS 22（Zenodo）→ Google Drive

在 Colab 中挂载 Google Drive，从 Zenodo 记录 [7262581](https://zenodo.org/records/7262581) 下载 **AMOS 2022** 相关文件到 `dataset/pretrain/Amos/download/`：

- `amos22.zip`：解压见下一格（输出在与 `download` 平级的 `unzip/`）
- `labeled_data_meta_0000_0599.csv`：元数据表，无需解压

下载使用 `wget -c` 与 `--show-progress`，支持断点续传；失败项会在末尾汇总。

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
import subprocess

DEST = "/content/drive/MyDrive/dataset/pretrain/Amos/download"
os.makedirs(DEST, exist_ok=True)

WGET_CMD = ["wget", "-c", "--show-progress"]

URLS = [
    (
        "https://zenodo.org/records/7262581/files/amos22.zip?download=1",
        "amos22.zip",
    ),
    (
        "https://zenodo.org/records/7262581/files/labeled_data_meta_0000_0599.csv?download=1",
        "labeled_data_meta_0000_0599.csv",
    ),
]

n = len(URLS)
failed = []

for i, (url, filename) in enumerate(URLS, start=1):
    out_path = os.path.join(DEST, filename)
    print(f"\n[{i}/{n}] {filename}")
    print(f"    -> {out_path}")
    print(f"    {url}")
    try:
        subprocess.run([*WGET_CMD, "-O", out_path, url], check=True)
        print("    OK")
    except subprocess.CalledProcessError as e:
        failed.append((filename, url, e.returncode))
        print(f"    FAILED (exit {e.returncode})")

print("\n" + "=" * 60)
if failed:
    print(f"下载失败 {len(failed)} 个（共 {n} 个）：")
    for filename, url, code in failed:
        print(f"  - {filename}  exit={code}")
        print(f"    {url}")
    raise RuntimeError(f"{len(failed)} 个文件下载失败，见上方列表")
print(f"全部完成（{n} 个）。目录内容：")
print(sorted(os.listdir(DEST)))


[1/2] amos22.zip
    -> /content/drive/MyDrive/dataset/pretrain/Amos/download/amos22.zip
    https://zenodo.org/records/7262581/files/amos22.zip?download=1
    OK

[2/2] labeled_data_meta_0000_0599.csv
    -> /content/drive/MyDrive/dataset/pretrain/Amos/download/labeled_data_meta_0000_0599.csv
    https://zenodo.org/records/7262581/files/labeled_data_meta_0000_0599.csv?download=1
    OK

全部完成（2 个）。目录内容：
['amos22.zip', 'labeled_data_meta_0000_0599.csv']


In [3]:
# 解压：仅处理 download 下的 .zip；每个 zip 解压到 unzip/<与压缩包同名的目录>/
import os
import zipfile

DOWNLOAD = "/content/drive/MyDrive/dataset/pretrain/Amos/download"
UNZIP_ROOT = "/content/drive/MyDrive/dataset/pretrain/Amos/unzip"
os.makedirs(UNZIP_ROOT, exist_ok=True)

zip_names = sorted(f for f in os.listdir(DOWNLOAD) if f.lower().endswith(".zip"))
failed = []

for i, zname in enumerate(zip_names, start=1):
    zpath = os.path.join(DOWNLOAD, zname)
    out_dir = os.path.join(UNZIP_ROOT, os.path.splitext(zname)[0])
    os.makedirs(out_dir, exist_ok=True)
    print(f"\n[{i}/{len(zip_names)}] {zname}")
    print(f"    -> {out_dir}")
    try:
        with zipfile.ZipFile(zpath, "r") as zf:
            bad = zf.testzip()
            if bad is not None:
                raise zipfile.BadZipFile(f"损坏条目: {bad}")
            zf.extractall(out_dir)
        print("    OK")
    except Exception as e:
        failed.append((zname, str(e)))
        print(f"    FAILED: {e}")

print("\n" + "=" * 60)
if failed:
    print(f"解压失败 {len(failed)} 个：")
    for zname, err in failed:
        print(f"  - {zname}: {err}")
    raise RuntimeError("部分 zip 解压失败，见上方列表")
print("全部 zip 解压完成。")
print("unzip 下顶层：", sorted(os.listdir(UNZIP_ROOT)))


[1/1] amos22.zip
    -> /content/drive/MyDrive/dataset/pretrain/Amos/unzip/amos22
    OK

全部 zip 解压完成。
unzip 下顶层： ['amos22']


In [4]:
import os
from collections import Counter, defaultdict

UNZIP_ROOT = "/content/drive/MyDrive/dataset/pretrain/Amos/unzip"

# 你可以按需调
MAX_SAMPLES_PER_EXT = 5
MAX_TREE_DEPTH = 4
MAX_ITEMS_PER_DIR = 25


def norm_rel(path, root):
    return os.path.relpath(path, root).replace("\\", "/")


def walk_summary(root):
    ext_counter = Counter()
    dir_file_counter = {}
    samples_by_ext = defaultdict(list)
    nii_like = []
    nii_gz = []
    json_files = []
    csv_files = []
    txt_files = []

    all_dirs = 0
    all_files = 0

    for dirpath, dirnames, filenames in os.walk(root):
        all_dirs += len(dirnames)
        all_files += len(filenames)
        rel_dir = norm_rel(dirpath, root)
        dir_file_counter[rel_dir] = len(filenames)

        for fn in filenames:
            full = os.path.join(dirpath, fn)
            rel = norm_rel(full, root)

            if fn.lower().endswith(".nii.gz"):
                ext = ".nii.gz"
            else:
                ext = os.path.splitext(fn)[1].lower() or "(no_ext)"

            ext_counter[ext] += 1

            if len(samples_by_ext[ext]) < MAX_SAMPLES_PER_EXT:
                samples_by_ext[ext].append(rel)

            if fn.lower().endswith(".nii.gz") or fn.lower().endswith(".nii"):
                nii_like.append(rel)
            if fn.lower().endswith(".nii.gz"):
                nii_gz.append(rel)
            if fn.lower().endswith(".json"):
                json_files.append(rel)
            if fn.lower().endswith(".csv"):
                csv_files.append(rel)
            if fn.lower().endswith(".txt"):
                txt_files.append(rel)

    return {
        "ext_counter": ext_counter,
        "samples_by_ext": samples_by_ext,
        "dir_file_counter": dir_file_counter,
        "all_dirs": all_dirs,
        "all_files": all_files,
        "nii_like": nii_like,
        "nii_gz": nii_gz,
        "json_files": json_files,
        "csv_files": csv_files,
        "txt_files": txt_files,
    }


def print_tree(path, root, prefix="", depth=0, max_depth=4, max_items=25):
    name = os.path.basename(path.rstrip("/")) or path
    if depth == 0:
        print(f"{name}/")

    if depth >= max_depth:
        print(f"{prefix}... (depth limit)")
        return

    try:
        entries = sorted(os.listdir(path))
    except Exception as e:
        print(f"{prefix}[无法访问] {e}")
        return

    shown = 0
    for entry in entries:
        if shown >= max_items:
            print(f"{prefix}... (remaining {len(entries) - shown} items)")
            break

        full = os.path.join(path, entry)
        if os.path.isdir(full):
            print(f"{prefix}{entry}/")
            print_tree(
                full,
                root,
                prefix=prefix + "    ",
                depth=depth + 1,
                max_depth=max_depth,
                max_items=max_items,
            )
        else:
            tag = ""
            low = entry.lower()
            if low.endswith(".nii.gz"):
                tag = " [nii.gz]"
            elif low.endswith(".json"):
                tag = " [json]"
            elif low.endswith(".csv"):
                tag = " [csv]"
            print(f"{prefix}{entry}{tag}")
        shown += 1


if not os.path.isdir(UNZIP_ROOT):
    print(f"目录不存在: {UNZIP_ROOT}")
else:
    info = walk_summary(UNZIP_ROOT)

    print("=" * 80)
    print("AMOS unzip 目录概览")
    print("根路径:", UNZIP_ROOT)
    print("=" * 80)

    print("\n[1] 总体规模")
    print(f"目录总数: {info['all_dirs']}")
    print(f"文件总数: {info['all_files']}")
    print(f".nii/.nii.gz 总数: {len(info['nii_like'])}")
    print(f"其中 .nii.gz 总数: {len(info['nii_gz'])}")
    print(f".json 总数: {len(info['json_files'])}")
    print(f".csv 总数: {len(info['csv_files'])}")
    print(f".txt 总数: {len(info['txt_files'])}")

    print("\n[2] 扩展名统计（前 15 个）")
    for ext, cnt in info["ext_counter"].most_common(15):
        print(f"{ext:12s} {cnt}")

    print("\n[3] 各扩展名样例路径")
    for ext, paths in sorted(info["samples_by_ext"].items(), key=lambda x: (-info["ext_counter"][x[0]], x[0])):
        print(f"\n{ext} ({info['ext_counter'][ext]}):")
        for p in paths:
            print(f"  - {p}")

    print("\n[4] 文件最多的目录（前 15 个）")
    dense_dirs = sorted(info["dir_file_counter"].items(), key=lambda x: (-x[1], x[0]))[:15]
    for d, n in dense_dirs:
        print(f"{d}: {n}")

    print("\n[5] .nii.gz 样例（前 20 个）")
    for p in info["nii_gz"][:20]:
        print("  -", p)

    print("\n[6] 顶层截断树")
    print_tree(UNZIP_ROOT, UNZIP_ROOT, max_depth=MAX_TREE_DEPTH, max_items=MAX_ITEMS_PER_DIR)

    print("\n完成。这个输出一般足够判断：")
    print("- 是不是按 images / labels / task / split 分目录")
    print("- 标注文件是 .nii.gz 还是 json")
    print("- 有没有 dataset.json / metadata / csv 之类的辅助文件")

AMOS unzip 目录概览
根路径: /content/drive/MyDrive/dataset/pretrain/Amos/unzip

[1] 总体规模
目录总数: 15
文件总数: 1200
.nii/.nii.gz 总数: 1191
其中 .nii.gz 总数: 1191
.json 总数: 1
.csv 总数: 0
.txt 总数: 0

[2] 扩展名统计（前 15 个）
.nii.gz      1191
(no_ext)     4
.md          2
.ds_store    2
.json        1

[3] 各扩展名样例路径

.nii.gz (1191):
  - amos22/amos22/labelsTr/amos_0001.nii.gz
  - amos22/amos22/labelsTr/amos_0584.nii.gz
  - amos22/amos22/labelsTr/amos_0162.nii.gz
  - amos22/amos22/labelsTr/amos_0588.nii.gz
  - amos22/amos22/labelsTr/amos_0113.nii.gz

(no_ext) (4):
  - amos22/amos22/.DS_Store
  - amos22/amos22/imagesTr/.DS_Store
  - amos22/__MACOSX/amos22/._imagesTr
  - amos22/__MACOSX/amos22/._imagesVa

.ds_store (2):
  - amos22/__MACOSX/amos22/._.DS_Store
  - amos22/__MACOSX/amos22/imagesTr/._.DS_Store

.md (2):
  - amos22/amos22/readme.md
  - amos22/__MACOSX/amos22/._readme.md

.json (1):
  - amos22/amos22/dataset.json

[4] 文件最多的目录（前 15 个）
amos22/amos22/imagesTr: 241
amos22/amos22/imagesTs: 240
amos22/amos22/labe